# 🐶🐱 Clasificador de Perros vs Gatos
**Proyecto Final - Inteligencia Artificial**

Este notebook implementa un clasificador de imágenes usando Transfer Learning con MobileNetV2.

### Pasos:
1. Descargar el dataset
2. Preprocesar imágenes
3. Construir el modelo (Transfer Learning)
4. Entrenar y evaluar
5. Interfaz interactiva con Gradio

## 1. Instalación de librerías

In [ ]:
# Instalar Gradio para la interfaz web
!pip install gradio -q

# Instalar kaggle para descargar el dataset
!pip install kaggle -q

print('✅ Librerías instaladas correctamente')

## 2. Descarga del Dataset

> **Instrucciones para obtener el dataset de Kaggle:**
> 1. Ve a https://www.kaggle.com/settings
> 2. Sección API → "Create New Token" → se descarga `kaggle.json`
> 3. Sube ese archivo cuando se te pida en la celda siguiente
>
> **Alternativa más fácil:** Usamos el dataset de TensorFlow directamente (sin Kaggle)

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import os

print(f'TensorFlow version: {tf.__version__}')

# Descargar dataset cats_vs_dogs directamente desde TensorFlow Datasets
# Esto descarga ~800MB - puede tardar unos minutos
(raw_train, raw_validation, raw_test), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)

print(f'\n✅ Dataset descargado:')
print(f'   Entrenamiento: {len(raw_train)} imágenes')
print(f'   Validación:    {len(raw_validation)} imágenes')
print(f'   Prueba:        {len(raw_test)} imágenes')
print(f'   Clases: {metadata.features["label"].names}')

## 3. Visualización de los datos

In [ ]:
# Ver algunas imágenes del dataset
class_names = ['Gato 🐱', 'Perro 🐶']

plt.figure(figsize=(12, 8))
for i, (image, label) in enumerate(raw_train.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(class_names[label.numpy()], fontsize=13)
    plt.axis('off')

plt.suptitle('Muestra del Dataset', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualización guardada como sample_images.png')

## 4. Preprocesamiento de Imágenes

In [ ]:
IMG_SIZE = 160  # MobileNetV2 recomienda 160x160
BATCH_SIZE = 32

def format_example(image, label):
    """Redimensiona y normaliza cada imagen."""
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = (image / 127.5) - 1  # Normalizar a [-1, 1] para MobileNetV2
    return image, label

# Aplicar preprocesamiento y crear batches
AUTOTUNE = tf.data.AUTOTUNE

train_batches = (
    raw_train
    .map(format_example, num_parallel_calls=AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

validation_batches = (
    raw_validation
    .map(format_example, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_batches = (
    raw_test
    .map(format_example, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
)

print(f'✅ Preprocesamiento completado')
print(f'   Tamaño de imagen: {IMG_SIZE}x{IMG_SIZE} px')
print(f'   Batch size: {BATCH_SIZE}')

# Verificar forma de los datos
for image_batch, label_batch in train_batches.take(1):
    print(f'   Shape de un batch: {image_batch.shape}')

## 5. Construcción del Modelo con Transfer Learning

Usamos **MobileNetV2** pre-entrenado en ImageNet (1.4 millones de imágenes, 1000 clases).
Solo necesitamos "fine-tunear" las últimas capas para nuestro problema binario.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Cargar MobileNetV2 SIN las capas finales (include_top=False)
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,   # No incluir las capas de clasificación originales
    weights='imagenet'   # Usar pesos pre-entrenados
)

# Congelar el modelo base (no entrenar esos pesos)
base_model.trainable = False

# Construir nuestro modelo encima
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),              # Regularización para evitar overfitting
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')  # Salida binaria: 0=gato, 1=perro
])

# Compilar el modelo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f'\n✅ Modelo construido con Transfer Learning (MobileNetV2)')
print(f'   Parámetros entrenables: {model.trainable_variables.__len__():,}')

## 6. Entrenamiento del Modelo

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Callbacks para mejorar el entrenamiento
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=3,          # Para si no mejora en 3 epochs
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Entrenar
print('🚀 Iniciando entrenamiento...')
print('   (Con GPU de Colab tarda ~5-10 minutos)\n')

history = model.fit(
    train_batches,
    epochs=10,
    validation_data=validation_batches,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Entrenamiento completado!')

## 7. Evaluación y Métricas

In [ ]:
# --- Curvas de entrenamiento ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(len(history.history['accuracy']))

ax1.plot(epochs_range, history.history['accuracy'], 'b-o', label='Entrenamiento')
ax1.plot(epochs_range, history.history['val_accuracy'], 'r-o', label='Validación')
ax1.set_title('Accuracy por Época', fontsize=14, fontweight='bold')
ax1.set_xlabel('Época')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history.history['loss'], 'b-o', label='Entrenamiento')
ax2.plot(epochs_range, history.history['val_loss'], 'r-o', label='Validación')
ax2.set_title('Loss por Época', fontsize=14, fontweight='bold')
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Curvas guardadas como training_curves.png')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Evaluar en conjunto de prueba
print('📊 Evaluando en datos de prueba...')
test_loss, test_accuracy = model.evaluate(test_batches, verbose=0)
print(f'\n🎯 Resultados finales en Test Set:')
print(f'   Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')
print(f'   Loss:     {test_loss:.4f}')

# Predicciones para métricas detalladas
y_pred = []
y_true = []

for images, labels in test_batches:
    predictions = model.predict(images, verbose=0)
    y_pred.extend((predictions > 0.5).astype(int).flatten())
    y_true.extend(labels.numpy())

# Reporte de clasificación
print('\n📋 Reporte de Clasificación:')
print(classification_report(y_true, y_pred, target_names=['Gato', 'Perro']))

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Gato 🐱', 'Perro 🐶'],
            yticklabels=['Gato 🐱', 'Perro 🐶'])
plt.title('Matriz de Confusión', fontsize=14, fontweight='bold')
plt.ylabel('Clase Real')
plt.xlabel('Clase Predicha')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Matriz de confusión guardada como confusion_matrix.png')

## 8. Comparación con Método Tradicional (SVM)

Comparamos nuestro modelo de Deep Learning contra un SVM clásico para justificar el uso de IA moderna.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

print('⏳ Preparando datos para SVM (puede tardar unos minutos)...')

# Extraer features aplanando las imágenes (método tradicional)
# Usamos un subconjunto pequeño para que sea manejable
X_train_svm, y_train_svm = [], []
X_test_svm, y_test_svm = [], []

# Tomar 2000 imágenes para entrenamiento SVM
count = 0
for images, labels in train_batches:
    for img, lbl in zip(images.numpy(), labels.numpy()):
        # Reducir resolución para SVM
        img_small = tf.image.resize(img, (32, 32)).numpy()
        X_train_svm.append(img_small.flatten())
        y_train_svm.append(lbl)
        count += 1
        if count >= 2000:
            break
    if count >= 2000:
        break

count = 0
for images, labels in test_batches:
    for img, lbl in zip(images.numpy(), labels.numpy()):
        img_small = tf.image.resize(img, (32, 32)).numpy()
        X_test_svm.append(img_small.flatten())
        y_test_svm.append(lbl)
        count += 1
        if count >= 500:
            break
    if count >= 500:
        break

X_train_svm = np.array(X_train_svm)
X_test_svm = np.array(X_test_svm)

# Normalizar
scaler = StandardScaler()
X_train_svm = scaler.fit_transform(X_train_svm)
X_test_svm = scaler.transform(X_test_svm)

# Entrenar SVM
print('🔧 Entrenando SVM...')
svm = SVC(kernel='rbf', C=1.0, random_state=42)
svm.fit(X_train_svm, y_train_svm)

svm_accuracy = accuracy_score(y_test_svm, svm.predict(X_test_svm))
print(f'\n📊 Comparación de Métodos:')
print(f'   SVM (método tradicional):        {svm_accuracy*100:.2f}%')
print(f'   CNN + Transfer Learning (nuestro): {test_accuracy*100:.2f}%')
print(f'   Mejora: +{(test_accuracy - svm_accuracy)*100:.2f}%')

# Gráfica comparativa
methods = ['SVM\n(Tradicional)', 'MobileNetV2\n(Transfer Learning)']
accuracies = [svm_accuracy * 100, test_accuracy * 100]
colors = ['#FF6B6B', '#4ECDC4']

plt.figure(figsize=(8, 6))
bars = plt.bar(methods, accuracies, color=colors, width=0.5, edgecolor='black')
plt.ylim([0, 100])
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Comparación: SVM vs Transfer Learning', fontsize=14, fontweight='bold')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparación guardada como comparison.png')

## 9. Predicciones en Imágenes de Ejemplo

In [ ]:
def predict_image(image, label=None):
    """Predice si una imagen es perro o gato."""
    img_array = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    img_array = tf.cast(img_array, tf.float32)
    img_array = (img_array / 127.5) - 1
    img_array = tf.expand_dims(img_array, 0)
    
    prediction = model.predict(img_array, verbose=0)[0][0]
    predicted_class = 'Perro 🐶' if prediction > 0.5 else 'Gato 🐱'
    confidence = prediction if prediction > 0.5 else 1 - prediction
    return predicted_class, confidence

# Mostrar predicciones en imágenes del test set
plt.figure(figsize=(15, 10))
correct = 0
total = 12

test_iter = iter(raw_test)
for i in range(total):
    image, label = next(test_iter)
    pred_class, confidence = predict_image(image)
    true_class = class_names[label.numpy()]
    is_correct = (pred_class.startswith('Perro') and label.numpy() == 1) or \
                 (pred_class.startswith('Gato') and label.numpy() == 0)
    if is_correct:
        correct += 1
    
    plt.subplot(3, 4, i + 1)
    plt.imshow(image)
    color = 'green' if is_correct else 'red'
    plt.title(f'Pred: {pred_class}\nConf: {confidence:.1%}\nReal: {true_class}',
              color=color, fontsize=9)
    plt.axis('off')

plt.suptitle(f'Predicciones en Test Set ({correct}/{total} correctas)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Predicciones guardadas como predictions.png')

## 10. Interfaz Web Interactiva con Gradio

Una app web donde puedes subir CUALQUIER foto y el modelo la clasifica en tiempo real.

In [ ]:
import gradio as gr
from PIL import Image

def classify_image(input_image):
    """Función principal de la interfaz Gradio."""
    # Preprocesar
    img = tf.image.resize(input_image, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = (img / 127.5) - 1
    img = tf.expand_dims(img, 0)
    
    # Predecir
    prediction = model.predict(img, verbose=0)[0][0]
    
    # Formatear resultado
    dog_prob = float(prediction)
    cat_prob = 1 - dog_prob
    
    return {
        "🐱 Gato": cat_prob,
        "🐶 Perro": dog_prob
    }

# Crear la interfaz
demo = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(label="Sube una foto de perro o gato"),
    outputs=gr.Label(num_top_classes=2, label="Predicción"),
    title="🐶🐱 Clasificador de Perros vs Gatos",
    description="Sube cualquier imagen y el modelo dirá si es un perro o un gato usando Transfer Learning con MobileNetV2.",
    theme=gr.themes.Soft()
)

# Lanzar la app (se abrirá un link público de Gradio)
demo.launch(share=True, debug=True)
print('\n✅ Interfaz lanzada! Usa el link de arriba para probarla.')

## 11. Guardar el Modelo

In [ ]:
# Guardar el modelo entrenado
model.save('cats_vs_dogs_model.h5')
print('✅ Modelo guardado como cats_vs_dogs_model.h5')

# Resumen final
print('\n' + '='*50)
print('📊 RESUMEN FINAL DEL PROYECTO')
print('='*50)
print(f'Modelo:          MobileNetV2 + Transfer Learning')
print(f'Dataset:         Microsoft Cats vs Dogs (TF Datasets)')
print(f'Accuracy final:  {test_accuracy*100:.2f}%')
print(f'vs SVM baseline: {svm_accuracy*100:.2f}%')
print(f'Mejora:          +{(test_accuracy - svm_accuracy)*100:.2f}%')
print('='*50)
print('\nArchivos generados:')
print('  📁 cats_vs_dogs_model.h5  - Modelo entrenado')
print('  📁 sample_images.png      - Muestra del dataset')
print('  📁 training_curves.png    - Curvas de entrenamiento')
print('  📁 confusion_matrix.png   - Matriz de confusión')
print('  📁 comparison.png         - Comparación SVM vs CNN')
print('  📁 predictions.png        - Ejemplos de predicciones')